In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch

In [ ]:
data = load_iris(as_frame=True)

df = data.frame

X = data.data
y = data.target
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")
# scaling
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

# converting y to array
y_train = np.array(y_train)
y_test = np.array(y_test)

# Convert to tensors with proper float32 dtype
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).long()
y_test_tensor = torch.from_numpy(y_test).long()

Training shape: (105, 4)
Testing shape: (45, 4)


In [ ]:
class NN():
    def __init__(self, X, num_classes=3):
        self.weights = torch.randn(X.shape[1], num_classes, dtype=torch.float32, requires_grad=True)
        self.bias = torch.zeros(num_classes, dtype=torch.float32, requires_grad=True)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        # Softmax for multi-class classification
        y_pred = torch.softmax(z, dim=1)
        return y_pred
    
    def loss(self, y_pred, y):
        # Cross-entropy loss for multi-class classification
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        # Convert target to one-hot encoding
        y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=3).float()
        loss = - (y_one_hot * torch.log(y_pred)).sum(dim=1)
        return loss.mean()

In [14]:
learning_rate = 0.1
epochs = 100

# initializing model

model = NN(X_train)

for epoch in range(epochs):
    # forward pass
    y_pred = model.forward(X_train_tensor)

    # loss calculation
    loss = model.loss(y_pred, y_train_tensor)

    # backward pass
    loss.backward()

    # tweaking the weights
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # Reset gradients to zero for next iteration
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1}, loss: {loss.item():.4f}")

Epoch: 10, loss: 0.6063
Epoch: 20, loss: 0.5236
Epoch: 30, loss: 0.4712
Epoch: 40, loss: 0.4352
Epoch: 50, loss: 0.4083
Epoch: 60, loss: 0.3868
Epoch: 70, loss: 0.3687
Epoch: 80, loss: 0.3529
Epoch: 90, loss: 0.3390
Epoch: 100, loss: 0.3265


In [17]:
# Model Evaluation
with torch.no_grad():
    # Training accuracy
    train_pred = model.forward(X_train_tensor)
    train_predictions = torch.argmax(train_pred, dim=1)
    train_accuracy = (train_predictions == y_train_tensor).float().mean()
    
    # Test accuracy
    test_pred = model.forward(X_test_tensor)
    test_predictions = torch.argmax(test_pred, dim=1)
    test_accuracy = (test_predictions == y_test_tensor).float().mean()

print(f"Training Accuracy: {train_accuracy.item():.4f}")
print(f"Test Accuracy: {test_accuracy.item():.4f}")

# Show some predictions
print("\nFirst 10 predictions vs actual:")
for i in range(10):
    print(f"Predicted: {train_predictions[i].item()}, Actual: {y_train_tensor[i].item()}")

Training Accuracy: 0.8571
Test Accuracy: 0.8444

First 10 predictions vs actual:
Predicted: 1, Actual: 1
Predicted: 1, Actual: 1
Predicted: 0, Actual: 0
Predicted: 2, Actual: 2
Predicted: 1, Actual: 1
Predicted: 2, Actual: 2
Predicted: 0, Actual: 0
Predicted: 0, Actual: 0
Predicted: 0, Actual: 0
Predicted: 2, Actual: 2
